# RT-LCOD — Run All trên T4: dữ liệu thật → train → distill → test → video UI

Notebook này bám đúng kiến trúc đã chốt:

**YOLOE-26M (open-vocabulary) + lightweight Attribute/Relation Grounding Head + NO_TARGET + Grounding DINO teacher chỉ dùng offline.**

Mục tiêu của notebook là **bấm Run All một lần**. Nó tự chuẩn bị một tập gRefCOCO thực, cache proposal/feature, train Stage 1, chạy teacher distillation Stage 2, chọn checkpoint trên validation, test held-out, nạp runtime cuối và hiển thị UI video có ô nhập prompt.

Mọi session đều in log: môi trường/GPU, dữ liệu, proposal recall, loss/accuracy, checkpoint, thời gian, VRAM và latency. Giữ toàn bộ output của notebook để dùng cho vòng tối ưu tiếp theo.


## Session 0 — Cấu hình một chỗ duy nhất

Các giá trị mặc định được chọn để một T4 16 GB có thể hoàn thành baseline đầu tiên trong thời gian hợp lý. Sau khi có log thật mới tăng dataset/epochs hoặc đổi input size.


In [ ]:
# ===== USER CONTROLS =====
TRAIN_SAMPLES = 2000
VAL_SAMPLES = 300
TEST_SAMPLES = 300
TEACHER_MAX_SAMPLES = 800     # Grounding DINO chạy offline trên tối đa số sample này
RUN_TEACHER = True            # False = bỏ Stage 2 để chạy nhanh hơn
REBUILD_CANDIDATE_CACHE = False

# Upload video của bạn lên Colab trước khi bấm Play (mặc định: /content/test.mp4).
# Nếu file chưa có, Run All vẫn hoàn tất và UI cuối notebook cho phép đổi đường dẫn.
VIDEO_PATH = "/content/test.mp4"
DEFAULT_PROMPT = "person"

REPOSITORY_URL = "https://github.com/dangtai111325/VLM.git"
BRANCH = "feature/rt-lcod-yoloe-student"

print("[CONFIG] train/val/test =", TRAIN_SAMPLES, VAL_SAMPLES, TEST_SAMPLES)
print("[CONFIG] teacher_max_samples =", TEACHER_MAX_SAMPLES, "run_teacher =", RUN_TEACHER)
print("[CONFIG] rebuild_candidate_cache =", REBUILD_CANDIDATE_CACHE)
print("[CONFIG] default video =", VIDEO_PATH)
print("[CONFIG] default prompt =", DEFAULT_PROMPT)


## Session 1 — Clone/cập nhật project, cài dependency, tạo RunAll orchestrator

Cell này tự tìm project. Nếu đang chạy Colab sạch, nó clone đúng branch. Sau đó cài package editable và bật widget manager. Cuối cell tạo `runner`, là đối tượng điều phối toàn bộ pipeline.


In [ ]:
from pathlib import Path
import os, subprocess, sys, time

HERE = Path.cwd().resolve()

def find_root():
    candidates = [
        HERE / "rt_lcod",
        HERE,
        HERE / "VLM" / "rt_lcod",
        Path("/content/VLM/rt_lcod"),
    ]
    return next((p for p in candidates if (p / "pyproject.toml").is_file()), None)

ROOT = find_root()
if ROOT is None:
    checkout = Path("/content/VLM")
    if checkout.exists():
        print("[SETUP] existing checkout found:", checkout)
    else:
        print("[SETUP] cloning", REPOSITORY_URL, "branch", BRANCH)
        subprocess.check_call(["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY_URL, str(checkout)])
    ROOT = checkout / "rt_lcod"

# A reused Colab VM must not silently execute an old checkout.
checkout = ROOT.parent
if (checkout / ".git").exists():
    dirty = subprocess.check_output(["git", "-C", str(checkout), "status", "--porcelain"], text=True).strip()
    if dirty:
        raise RuntimeError(f"Refusing to overwrite modified checkout: {checkout}. Use a fresh Colab runtime or commit/stash its changes.")
    print("[SETUP] updating checkout to", BRANCH)
    subprocess.check_call(["git", "-C", str(checkout), "fetch", "origin", BRANCH])
    subprocess.check_call(["git", "-C", str(checkout), "checkout", BRANCH])
    subprocess.check_call(["git", "-C", str(checkout), "pull", "--ff-only", "origin", BRANCH])
else:
    print("[SETUP] source is not a Git checkout; using the local files as provided")

if not (ROOT / "pyproject.toml").is_file():
    raise RuntimeError(f"RT-LCOD project not found: {ROOT}")

print("[SETUP] ROOT =", ROOT)
print("[SETUP] python =", sys.version.replace("\n", " "))
print("[SETUP] installing project dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--editable", f"{ROOT}[all]"])

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
    print("[SETUP] Colab custom widget manager enabled")
except Exception:
    print("[SETUP] not running in Colab or widget manager already available")

os.chdir(ROOT)
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from rt_lcod.runall import RunAll

runner = RunAll(
    ROOT,
    train_samples=TRAIN_SAMPLES,
    val_samples=VAL_SAMPLES,
    test_samples=TEST_SAMPLES,
    teacher_max_samples=TEACHER_MAX_SAMPLES,
    run_teacher=RUN_TEACHER,
    rebuild_candidate_cache=REBUILD_CANDIDATE_CACHE,
)
print("[SETUP] RunAll ready")


## Session 2 — T4 / CUDA preflight

Dừng ngay nếu CUDA không khả dụng. Log này cho biết GPU, VRAM, CUDA/PyTorch và config inference/training thực tế.


In [ ]:
runner.preflight()


## Session 3 — Dataset thật: gRefCOCO → canonical train/val/test

Tự tải metadata gRefCOCO và **chỉ các COCO image cần cho subset hiện tại**. V1 giữ single-target hoặc true no-target samples. Dataset được tách theo official split, sau đó validate schema và in thống kê/samples.


In [ ]:
runner.prepare_data()


## Session 4 — YOLOE proposal + frozen visual/text cache

Mỗi sample chạy YOLOE-26M-seg một lần để lấy candidate. Sau đó MobileNetV3-Small + ROIAlign lấy region features và MiniLM lấy attribute/relation embedding. Cache giúp 10 epoch training không phải chạy lại YOLOE.

Quan trọng nhất ở đây là `positive_proposal_recall_at_iou`: nếu YOLOE không propose target thì grounding head không thể cứu.


In [ ]:
runner.cache_candidates()


## Session 5 — Stage 1: train grounding student trên dữ liệu thật

YOLOE, MobileNet và MiniLM vẫn frozen. Chỉ train lightweight grounding head: visual projection, attribute score, relation score, candidate fusion và NO_TARGET. Trainer lưu `last.pt`, `best.pt`, checkpoint mỗi epoch, metrics và timing.

Sau train, validation chạy ngay và in metric.


In [ ]:
runner.train_stage1()
stage1_val = runner.evaluate_stage1_validation()
print("[NOTEBOOK] Stage 1 validation:")
print(stage1_val)


## Session 6 — Grounding DINO teacher cache (offline)

Teacher **không chạy khi deploy video**. Nó chỉ tạo soft distribution trên một subset train để student học thêm cross-modal grounding. Vì T4 có giới hạn thời gian, mặc định teacher dùng tối đa `TEACHER_MAX_SAMPLES`.


In [ ]:
runner.cache_teacher_outputs()


## Session 7 — Stage 2: Knowledge Distillation

Khởi tạo student từ `best.pt` Stage 1 nhưng bắt đầu optimizer/scheduler mới. Các sample có teacher target dùng KL distillation; sample chưa có teacher vẫn dùng supervised losses bình thường.


In [ ]:
runner.train_stage2()


## Session 8 — Chọn model cuối bằng VALIDATION, không nhìn TEST

So Stage 1 và Stage 2 trên validation bằng composite `0.70 * target_top1 + 0.30 * no_target_accuracy`. Checkpoint tốt hơn được chọn làm final model.


In [ ]:
FINAL_CHECKPOINT = runner.select_final_on_validation()
print("[NOTEBOOK] FINAL_CHECKPOINT =", FINAL_CHECKPOINT)


## Session 9 — Held-out TEST một lần

Chỉ sau khi model đã được chọn trên validation mới đánh giá test. Log này là kết quả khách quan đầu tiên để so sánh các lần tối ưu sau.


In [ ]:
final_test = runner.final_test()
print("[NOTEBOOK] FINAL TEST:")
print(final_test)


## Session 10 — Load runtime cuối

Nạp checkpoint đã chọn + YOLOE-26M + frozen region/text encoders lên T4. Sau cell này model đã sẵn sàng cho video.


In [ ]:
runtime = runner.load_runtime()


## Session 11 — Video player + ô prompt

Đây là cell cuối của **Run All**. UI xuất hiện ngay cả khi `VIDEO_PATH` mặc định chưa tồn tại.

1. Nhập đường dẫn video vào ô **Video** (ví dụ `/content/my_video.mp4`).
2. Nhập prompt V1, ví dụ `red car`, `person next to bicycle`, `bottle left of laptop`.
3. Bấm **Play**.
4. Có thể đổi prompt trong lúc video đang chạy.
5. `Infer every = 1` là raw per-frame model; tăng lên 2–5 là chế độ thực dụng nhanh hơn.

Log bên dưới video ghi detector/region/head latency, candidate count, confidence, VRAM và cuối video có mean/p50/p95/p99/FPS.


In [ ]:
player = runner.show_video_ui(VIDEO_PATH, DEFAULT_PROMPT)
print("[NOTEBOOK] Run All completed. The interactive video UI above is ready.")
